# CNN for Solving a 5-Bus Power System Problem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enliten/IEEE-course_AI-for-Power-Energy-Systems/blob/main/03_CNN_for_5_bus_system/CNN_for_5_bus_system.ipynb)

This Jupyter Notebook implements a simple **Convolutional Neural Network (CNN)** using TensorFlow/Keras to solve a 5-bus power system problem. The models are trained on a **Monte-Carlo dataset**: the load demand is perturbed many times, the AC power flow is solved for each case, and the network learns to predict the solved bus voltage magnitudes and line power flows from the (leakage-free) load and generation inputs. The test system is the modified PJM 5-bus case (`case5` in pandapower), based on F. Li and R. Bo, "Small Test Systems for Power System Economic Studies," DOI: [10.1109/PES.2010.5589973](https://doi.org/10.1109/PES.2010.5589973).

## Table of Contents:
1. [Load the 5-bus power system from pandapower](#step-1-load-the-5-bus-power-system-from-pandapower)
2. [Generate training data by Monte-Carlo sampling of the load demand](#step-2-generate-training-data-by-monte-carlo-sampling-of-the-load-demand)
3. [Define the CNN models](#step-3-define-the-cnn-models)
4. [Train the models](#step-4-train-the-model-using-power-system-data)
5. [Predict on held-out test data](#step-5-predict-results-for-new-input-data)
6. [Compare predictions and visualize model performance](#step-6-compare-predictions-for-different-training-epochs-and-filter-sizes)


In [ ]:
# Install required packages
!pip install pandapower tensorflow matplotlib numpy

In [ ]:
import pandapower as pp
import pandas as pd
import pandapower.networks as nw
import pandapower.plotting as pplot
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

## Step 1. Load the 5-bus power system from pandapower
CASE5  Power flow data for modified 5 bus, 5 gen case based on PJM 5-bus system

In [ ]:
# Load the 5-bus power system network
net = nw.case5()
print(net)

In [ ]:
# Run power flow analysis
net.bus['max_vm_pu'] = 1.10
net.bus['min_vm_pu'] = 0.90
pp.runpp(net)

# Check available columns for loads and generators
print("Load columns:", net.load.columns)
print("Gen columns:", net.gen.columns)

## Step 2. Generate training data by Monte-Carlo sampling of the load demand

A single power-flow snapshot only gives 5 bus values, which is far too little (and the voltage would appear on both sides of the model). Instead we build a proper dataset: for each of `N_SAMPLES` operating points we perturb the **load demand** with Gaussian noise, feed it to `pp.runpp`, and record the solved result.

**Inputs `X` — known *before* the solve, so there is no label leakage.** For every bus: `[Pd, Qd, Pg_set, V_set]`
- `Pd`, `Qd` — load active / reactive demand (the Monte-Carlo perturbed quantities);
- `Pg_set` — generator active-power setpoint;
- `V_set` — generator / slack **voltage setpoint** (a control input, *not* the solved bus voltage).

The solved voltage magnitude `res_bus.vm_pu` is deliberately **excluded from the input** so the model cannot simply copy the answer. (Generator reactive power `Qg` is also excluded — it is a *result* of the power flow, not an input.)

**Outputs `Y` — solved by the power flow:** bus voltage magnitudes (`res_bus.vm_pu`) and line power flows (`res_line.p_from_mw`, `res_line.q_from_mvar`).


In [ ]:
# ---- Monte Carlo data generation -------------------------------------------
# Perturb the LOAD DEMAND with Gaussian noise, solve the AC power flow for each
# sample, and collect leakage-free inputs (X) and solved outputs (Y).
rng = np.random.default_rng(42)   # reproducible sampling
N_SAMPLES = 100                   # number of Monte Carlo power-flow runs
LOAD_STD  = 0.10                  # load noise std = 10% of the nominal demand

n_bus  = len(net.bus)
n_line = len(net.line)

# Nominal (base-case) load setpoints we perturb around
base_p = net.load.p_mw.values.copy()
base_q = net.load.q_mvar.values.copy()

# Constant per-bus generation setpoints (known before solving -> valid inputs)
Pg_set = np.zeros(n_bus)
np.add.at(Pg_set, net.gen.bus.values, net.gen.p_mw.values)
V_set = np.zeros(n_bus)
V_set[net.gen.bus.values]      = net.gen.vm_pu.values
V_set[net.ext_grid.bus.values] = net.ext_grid.vm_pu.values

X_list, Yv_list, Ypf_list = [], [], []
for _ in range(N_SAMPLES):
    # 1) Sample a new load demand ~ N(nominal, (10% * nominal)^2); keep P > 0
    net.load.p_mw   = np.clip(base_p * rng.normal(1.0, LOAD_STD, size=base_p.size), 1e-3, None)
    net.load.q_mvar = base_q * rng.normal(1.0, LOAD_STD, size=base_q.size)

    # 2) Solve the AC power flow for this operating point
    try:
        pp.runpp(net)
    except Exception:
        continue  # skip the rare non-converging sample

    # 3) Per-bus INPUT features, all known BEFORE the solve -> no vm_pu leakage
    Pd = np.zeros(n_bus); np.add.at(Pd, net.load.bus.values, net.load.p_mw.values)
    Qd = np.zeros(n_bus); np.add.at(Qd, net.load.bus.values, net.load.q_mvar.values)
    X_list.append(np.column_stack([Pd, Qd, Pg_set, V_set]))   # (n_bus, 4)

    # 4) OUTPUT targets solved by the power flow
    Yv_list.append(net.res_bus.vm_pu.values)                            # (n_bus,)
    Ypf_list.append(net.res_line[['p_from_mw', 'q_from_mvar']].values)  # (n_line, 2)

# Restore the base case so `net` is left in a clean state
net.load.p_mw, net.load.q_mvar = base_p, base_q
pp.runpp(net)

X            = np.array(X_list)                              # (N, n_bus, 4)  -> CNN input
Y_voltage    = np.array(Yv_list)                            # (N, n_bus)     -> voltage target
Y_power_flow = np.array(Ypf_list).reshape(len(Ypf_list), -1)  # (N, n_line*2) -> P/Q target

print(f"Converged samples : {len(X)} / {N_SAMPLES}")
print("X (inputs)        :", X.shape, "-> [Pd, Qd, Pg_set, V_set] per bus (no solved vm_pu)")
print("Y_voltage         :", Y_voltage.shape)
print("Y_power_flow      :", Y_power_flow.shape, f"-> {n_line} lines x (P, Q)")

# ---- Shuffle, split 80/20, standardise the inputs --------------------------
perm = rng.permutation(len(X))
X, Y_voltage, Y_power_flow = X[perm], Y_voltage[perm], Y_power_flow[perm]

n_train = int(0.8 * len(X))
X_train,   X_test    = X[:n_train],            X[n_train:]
Yv_train,  Yv_test   = Y_voltage[:n_train],    Y_voltage[n_train:]
Ypf_train, Ypf_test  = Y_power_flow[:n_train], Y_power_flow[n_train:]

# Standardise inputs per feature (constant setpoint columns keep std = 1)
feat_mean = X_train.reshape(-1, X_train.shape[-1]).mean(axis=0)
feat_std  = X_train.reshape(-1, X_train.shape[-1]).std(axis=0)
feat_std[feat_std == 0] = 1.0
X_train = (X_train - feat_mean) / feat_std
X_test  = (X_test  - feat_mean) / feat_std


## Step 3. Define the CNN models

Each **operating point** is now one training sample shaped `5 buses × 4 features`; a `Conv1D` slides over the 5 buses with the 4 electrical features (`Pd, Qd, Pg_set, V_set`) as channels. The solved voltage is *not* part of the input.

### Bus voltage (compare training epochs / filter counts)
- Model 1 — epochs=50, 24 filters
- Model 2 — epochs=50, 240 filters
- Model 3 — epochs=200, 240 filters

Output layer = one node per bus voltage (`n_bus_out`). *Note:* in `case5` the PV/slack buses are held at their voltage setpoint, so only the single PQ-bus voltage actually moves — the line flows below are the richer prediction target.

### Power Flow
- Model 4 — epochs=200, 24 filters; output layer = 6 lines × (P, Q) = `n_line_out`.

<figure>
  <img src="https://assets.zilliz.com/The_Architecture_of_Convolutional_Neural_Networks_8263469ad1.png" width="450">
</figure>


In [ ]:
input_shape = (X.shape[1], X.shape[2])   # (n_bus = 5 buses, n_features = 4 channels)
n_bus_out   = Y_voltage.shape[1]         # 5 bus-voltage outputs
n_line_out  = Y_power_flow.shape[1]      # 12 = 6 lines x (P, Q)

# ---- Bus-voltage models: compare epochs / filter counts -------------------
# Model 1 (epochs=50, 24 filters)
model_50 = keras.Sequential([
    layers.Conv1D(filters=24, kernel_size=2, activation='relu', input_shape=input_shape),
    layers.Conv1D(filters=24, kernel_size=2, activation='relu'),
    layers.Conv1D(filters=12, kernel_size=2, activation='relu'),
    layers.Flatten(),
    layers.Dense(12, activation='relu'),
    layers.Dense(n_bus_out, activation='linear')  # one output per bus voltage
])
model_50.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Model 2 (epochs=50, 240 filters)
model_50_v2 = keras.Sequential([
    layers.Conv1D(filters=240, kernel_size=2, activation='relu', input_shape=input_shape),
    layers.Conv1D(filters=240, kernel_size=2, activation='relu'),
    layers.Conv1D(filters=120, kernel_size=2, activation='relu'),
    layers.Flatten(),
    layers.Dense(12, activation='relu'),
    layers.Dense(n_bus_out, activation='linear')
])
model_50_v2.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Model 3 (epochs=200, 240 filters)
model_200 = keras.Sequential([
    layers.Conv1D(filters=240, kernel_size=2, activation='relu', input_shape=input_shape),
    layers.Conv1D(filters=240, kernel_size=2, activation='relu'),
    layers.Conv1D(filters=120, kernel_size=2, activation='relu'),
    layers.Flatten(),
    layers.Dense(12, activation='relu'),
    layers.Dense(n_bus_out, activation='linear')
])
model_200.compile(optimizer='adam', loss='mse', metrics=['mae'])

# ---- Power-flow model: predict the line flows P and Q ---------------------
# Model 4 (epochs=200, 24 filters)
model_power_flow = keras.Sequential([
    layers.Conv1D(filters=24, kernel_size=2, activation='relu', input_shape=input_shape),
    layers.Conv1D(filters=24, kernel_size=2, activation='relu'),
    layers.Conv1D(filters=12, kernel_size=2, activation='relu'),
    layers.Flatten(),
    layers.Dense(12, activation='relu'),
    layers.Dense(n_line_out, activation='linear')  # 6 lines x (P, Q)
])
model_power_flow.compile(optimizer='adam', loss='mse', metrics=['mae'])


## Step 4. Train the model using power system data

In [ ]:
# Voltage models are trained on Yv_train; the power-flow model on Ypf_train.
history_50    = model_50.fit(X_train, Yv_train, epochs=50, batch_size=16, validation_split=0.2, verbose=0)
history_50_v2 = model_50_v2.fit(X_train, Yv_train, epochs=50, batch_size=16, validation_split=0.2, verbose=0)
history_200   = model_200.fit(X_train, Yv_train, epochs=200, batch_size=16, validation_split=0.2, verbose=0)
history_pf    = model_power_flow.fit(X_train, Ypf_train, epochs=200, batch_size=16, validation_split=0.2, verbose=0)

print("Model 1 (50 ep,  24 f) final val_loss:", round(history_50.history['val_loss'][-1], 6))
print("Model 2 (50 ep, 240 f) final val_loss:", round(history_50_v2.history['val_loss'][-1], 6))
print("Model 3 (200 ep,240 f) final val_loss:", round(history_200.history['val_loss'][-1], 6))
print("Model 4 (power flow)   final val_loss:", round(history_pf.history['val_loss'][-1], 4))


## Step 5. Predict results for new input data

In [ ]:
# Predict on the held-out TEST operating points (unseen during training)
pred_v_50   = model_50.predict(X_test, verbose=0)
pred_v_50v2 = model_50_v2.predict(X_test, verbose=0)
pred_v_200  = model_200.predict(X_test, verbose=0)
pred_pf     = model_power_flow.predict(X_test, verbose=0)

print("Test voltage MAE    (Model 3, pu):", np.mean(np.abs(pred_v_200 - Yv_test)))
print("Test power-flow MAE (MW/MVAr)    :", np.mean(np.abs(pred_pf - Ypf_test)))


## Step 6. Compare predictions for different training epochs and filter sizes.

In [ ]:
# (a) Predicted vs actual bus voltages for one held-out operating point
i = 0
plt.figure(figsize=(10, 5))
plt.plot(Yv_test[i],     'bo--', label='Actual Voltage Magnitudes')
plt.plot(pred_v_50[i],   'r*-',  label='Predicted (Epochs=50, 24 filters)')
plt.plot(pred_v_50v2[i], 'g^-',  label='Predicted (Epochs=50, 240 filters)')
plt.plot(pred_v_200[i],  'y+-',  label='Predicted (Epochs=200, 240 filters)')
plt.xlabel('Bus Index')
plt.ylabel('Voltage Magnitude (pu)')
plt.title('Actual vs Predicted Voltage Magnitudes (held-out sample)')
plt.xticks(ticks=range(n_bus), labels=range(1, n_bus + 1))
plt.legend()
plt.show()

# (b) In case5 only the PQ bus voltage really varies; check the model tracks it
b = int(np.argmax(Yv_test.var(axis=0)))
plt.figure(figsize=(5, 5))
plt.scatter(Yv_test[:, b], pred_v_200[:, b], s=25)
lims = [Yv_test[:, b].min(), Yv_test[:, b].max()]
plt.plot(lims, lims, 'k--', label='ideal')
plt.xlabel(f'Actual Voltage at Bus {b + 1} (pu)')
plt.ylabel('Predicted Voltage (pu)')
plt.title('Model 3 tracks the varying PQ-bus voltage (test set)')
plt.legend()
plt.show()


In [ ]:
# (a) Predicted vs actual line flows for one held-out operating point
i = 0
pf_actual = Ypf_test[i].reshape(n_line, 2)   # columns: [P (MW), Q (MVAr)]
pf_pred   = pred_pf[i].reshape(n_line, 2)

plt.figure(figsize=(10, 5))
plt.plot(pf_actual[:, 0], 'bo--', label='Actual P (MW)')
plt.plot(pf_pred[:, 0],   'r*-',  label='Predicted P (MW)')
plt.xlabel('Line Index')
plt.ylabel('Active Power Flow (MW)')
plt.legend()
plt.title('Active Power Flow Prediction (held-out sample)')
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(pf_actual[:, 1], 'bo--', label='Actual Q (MVAR)')
plt.plot(pf_pred[:, 1],   'r*-',  label='Predicted Q (MVAR)')
plt.xlabel('Line Index')
plt.ylabel('Reactive Power Flow (MVAR)')
plt.legend()
plt.title('Reactive Power Flow Prediction (held-out sample)')
plt.show()

# (b) Overall fit across ALL test samples and all line flows
plt.figure(figsize=(5, 5))
plt.scatter(Ypf_test.ravel(), pred_pf.ravel(), s=10, alpha=0.5)
lims = [Ypf_test.min(), Ypf_test.max()]
plt.plot(lims, lims, 'k--', label='ideal')
plt.xlabel('Actual line flow (MW / MVAr)')
plt.ylabel('Predicted line flow')
plt.title('Power-flow model: predicted vs actual (test set)')
plt.legend()
plt.show()

# Print Predictions for the first test sample
print("Predicted line flows for the first test sample (P, Q per line):")
print(pf_pred)
